# 02 — Exploratory Data Analysis & Feature Engineering

**Trinity University Football Analytics**

This notebook takes the cleaned `all_df` from `01_data_cleaning.ipynb` and:
1. Engineers all features needed for the EP/EPA model
2. Explores trends across seasons, units, and game situations

**Input:** `all_df` from notebook 01

**Output:** `all_df` with engineered features — `drive_points`, `yards_to_go`, `goal_to_go`, `points`, `score_event`, `next_score`, `turnover_on_downs`, `game_id`

## 1. Imports

> **Note:** Run `01_data_cleaning.ipynb` first to generate `all_df`, then continue here.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# all_df should already be in memory from 01_data_cleaning.ipynb
# If running standalone, load from your exported Excel:
# all_df = pd.read_excel('/content/drive/MyDrive/TUFB_EPA/TUFB_ALL_games.xlsx')
print(f"all_df shape: {all_df.shape}")

## 2. Feature Engineering

### 2a. Drive Points
Calculates how many points each drive produced. Used as the target variable for the EP model.

In [ ]:
def calc_drive_points(game_df, series_col, series_val, odk_val):
    series_idx = game_df[
        (game_df[series_col] == series_val) & (game_df['odk'] == odk_val)
    ].index

    if len(series_idx) == 0:
        return 0

    last_play = game_df.loc[series_idx, 'play'].max()
    results = game_df.loc[series_idx, 'result'].fillna('').str.lower()

    k_after = game_df[
        (game_df['play'] > last_play) & (game_df['odk'] == 'K')
    ].head(2)
    k_play_types = k_after['play_type'].fillna('').str.strip().str.lower()
    k_results = k_after['result'].fillna('').str.strip().str.lower()

    if results.str.contains('def td').any():
        return -7
    if results.str.contains('td').any() and not results.str.contains('def td').any():
        return 7
    if (k_results.str.contains('good') & k_play_types.isin(['fg', 'fg block'])).any():
        return 3
    return 0


all_df['drive_points'] = 0.0

for (team, date), game_df in all_df.groupby(['team_name', 'game_date']):
    game_df = game_df.copy()

    for series_val in game_df.loc[
        (game_df['off_series'] != 0) & (game_df['odk'] == 'O'), 'off_series'
    ].unique():
        pts = calc_drive_points(game_df, 'off_series', series_val, 'O')
        idx = all_df[
            (all_df['team_name'] == team) & (all_df['game_date'] == date) &
            (all_df['off_series'] == series_val)
        ].index
        all_df.loc[idx, 'drive_points'] = pts

    for series_val in game_df.loc[
        (game_df['def_series'] != 0) & (game_df['odk'] == 'D'), 'def_series'
    ].unique():
        pts = calc_drive_points(game_df, 'def_series', series_val, 'D')
        idx = all_df[
            (all_df['team_name'] == team) & (all_df['game_date'] == date) &
            (all_df['def_series'] == series_val)
        ].index
        all_df.loc[idx, 'drive_points'] = pts

    for i, row in game_df[game_df['odk'] == 'K'].iterrows():
        result = str(row['result']).lower()
        play_type = str(row['play_type']).lower()
        if result == 'td':
            all_df.loc[i, 'drive_points'] = -7 if play_type in ['punt', 'ko'] else 7 if play_type in ['punt rec', 'ko rec'] else 0
        elif 'good' in result and play_type in ['fg', 'fg block']:
            all_df.loc[i, 'drive_points'] = 3
        else:
            all_df.loc[i, 'drive_points'] = 0

print("Drive points distribution:")
print(all_df['drive_points'].value_counts().sort_index())

### 2b. yards_to_go & goal_to_go

In [ ]:
# yards_to_go: distance from the end zone (own side = 100 + yard_ln)
all_df['yards_to_go'] = np.where(
    all_df['yard_ln'] < 0,
    100 + all_df['yard_ln'],
    all_df['yard_ln']
)

# goal_to_go: offense inside 10 yards of end zone
all_df['goal_to_go'] = (
    (all_df['yards_to_go'] <= 10) & (all_df['odk'] == 'O')
).astype(int)

print(all_df[['yard_ln', 'yards_to_go', 'goal_to_go']].head(10))

### 2c. Points, Game ID, Score Variables, Turnover on Downs, Score Event, Next Score

In [ ]:
# Standardize column names and key values
all_df.columns = (
    all_df.columns.str.strip().str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
)
for col in ['odk', 'result', 'play_type']:
    if col in all_df.columns:
        all_df[col] = all_df[col].astype(str).str.strip().str.lower()

off_td = ['rush, td', 'complete, td', 'scramble, td']
def_td = ['interception, def td', 'fumble, def td', 'sack, fumble, def td']

def assign_points(row):
    result = row['result']
    odk = row['odk']
    play_type = row['play_type']

    if play_type == '2 pt.':
        return 2 if result == 'good' else 0
    if play_type == '2 pt. defend':
        return -2 if result == 'good' else 0
    if result in off_td:
        return 6 if odk == 'o' else -6
    if result in def_td:
        return 6 if odk == 'd' else -6
    if play_type == 'fg':
        return 3 if result == 'good' else -6 if 'td' in result else 0
    if play_type == 'fg block':
        return -3 if result == 'good' else 6 if 'td' in result else 0
    if play_type == 'extra pt.':
        return 1 if result == 'good' else -2 if 'td' in result else 0
    if play_type == 'extra pt. block':
        return -1 if result == 'good' else 2 if 'td' in result else 0
    if play_type == 'ko rec' and 'td' in result:
        return 6
    if play_type == 'punt rec' and 'td' in result:
        return 6
    if play_type in ['ko', 'punt'] and 'td' in result:
        return -6
    if result in ['rush, safety', 'sack, safety', 'safety']:
        return -2 if odk == 'o' else 2
    return 0

all_df['points'] = all_df.apply(assign_points, axis=1)

# Game ID
all_df['game_id'] = (
    all_df['game_date'].astype(str) + "_" +
    all_df[['team_name', 'opp_name']].apply(
        lambda x: "_".join(sorted([x['team_name'], x['opp_name']])), axis=1
    )
)

# Sort and running point differential
all_df['play'] = pd.to_numeric(all_df['play'], errors='coerce')
all_df = all_df.sort_values(['game_id', 'play']).reset_index(drop=True)
all_df['point_differential'] = all_df.groupby('game_id')['points'].cumsum()

# Score variables
all_df['off_score'] = all_df['team_pts']
all_df['def_score'] = all_df['opp_pts']

# Turnover on downs
all_df['next_odk'] = all_df.groupby('game_id')['odk'].shift(-1)
all_df['turnover_on_downs'] = (
    (all_df['dn'] == 4) & (all_df['odk'] != all_df['next_odk'])
).astype(int)

print("Points distribution:")
print(all_df['points'].value_counts().sort_index())

In [ ]:
# Remove penalty rows, then build score_event and next_score
all_df = all_df[all_df['result'] != 'penalty'].reset_index(drop=True)

off_td_results = ['rush, td', 'complete, td', 'scramble, td', 'td']
def_td_results  = ['interception, def td', 'fumble, def td', 'sack, fumble, def td', 'block, def td']

def assign_score_event(row):
    result    = str(row['result']).strip().lower()
    play_type = str(row['play_type']).strip().lower()
    if result in off_td_results:
        return 7
    if result in def_td_results:
        return -7
    if 'td' in result:
        return -7 if 'def' in result else 7
    if play_type in ['fg', 'fg block'] and result == 'good':
        return 3
    if play_type in ['punt rec', 'ko rec'] and result == 'td':
        return 7
    if play_type in ['punt', 'ko'] and result == 'td':
        return -7
    return 0

all_df['score_event'] = all_df.apply(assign_score_event, axis=1)

all_df['score_event_nonzero'] = all_df['score_event'].replace(0, np.nan)
all_df['next_score'] = (
    all_df.groupby('game_id')['score_event_nonzero']
    .transform(lambda x: x.shift(-1).bfill())
).fillna(0)
all_df = all_df.drop(columns=['score_event_nonzero'])

print("Score event distribution:")
print(all_df['score_event'].value_counts().sort_index())
print(f"\nRows after removing penalties: {len(all_df)}")

## 3. Exploratory Analysis

### 3a. Play Type Breakdown by Unit

In [ ]:
od_df = all_df[all_df['odk'].isin(['o', 'd'])].copy()

play_type_counts = od_df.groupby(['odk', 'play_type']).size().reset_index(name='count')
play_type_counts = play_type_counts.sort_values(['odk', 'count'], ascending=[True, False])
print(play_type_counts.to_string(index=False))

### 3b. Drive Results by Unit

In [ ]:
print("Offensive drive points distribution:")
print(all_df[all_df['odk'] == 'o']['drive_points'].value_counts().sort_index())

print("\nDefensive drive points distribution:")
print(all_df[all_df['odk'] == 'd']['drive_points'].value_counts().sort_index())

### 3c. Explosive Play Rate

In [ ]:
explosive_rate = od_df.groupby('odk')['explosive'].mean().round(3)
print("Explosive play rate by unit:")
print(explosive_rate)

### 3d. Yards Per Play by Down

In [ ]:
od_df['gn_ls'] = pd.to_numeric(od_df['gn_ls'], errors='coerce')
print("Average yards gained by down (Offense):")
print(od_df[od_df['odk'] == 'o'].groupby('dn')['gn_ls'].mean().round(2))

### 3e. Win Rate by Home/Away

In [ ]:
game_level = all_df.drop_duplicates('game_id')[['game_id', 'home_away', 'win']]
print("Win rate by home/away:")
print(game_level.groupby('home_away')['win'].mean().round(3))

## 4. Export (Optional)

Uncomment to save the full engineered dataframe before modeling.

In [ ]:
# all_df.to_excel('/content/drive/MyDrive/TUFB_EPA/TUFB_ALL_games.xlsx', sheet_name='PlayList', index=False)
# print('Exported successfully.')